In [ ]:
def classify_with_rag(ticket: str, top_k: int = 2) -> Dict:
    """Classify ticket using RAG pipeline."""
    
    # Step 1: Retrieve relevant documents
    retrieved_docs = retrieve_documents(ticket, top_k=top_k)
    context = "\n".join([doc for doc, _ in retrieved_docs])
    
    # Step 2: Create prompt with context
    prompt = f"""You are a support ticket classifier. Based on the ticket and context, classify it.

Ticket: {ticket}

Context:
{context}

Categories: Account, Billing, Shipping, Product, Technical

Classification:"""
    
    # Step 3: Simulate LLM response (in practice, would call actual LLM)
    # For now, extract category from context
    for doc, score in retrieved_docs:
        for _, category in knowledge_base:
            if category in doc:
                confidence = score
                break
    
    result = {
        "ticket": ticket,
        "classification": categories[np.argsort(cosine_similarity(
            vectorizer.transform([ticket]), doc_embeddings)[0])[::-1][0]],
        "confidence": 0.85,
        "retrieved_docs": [(doc[:50] + "...", score) for doc, score in retrieved_docs],
        "prompt": prompt
    }
    
    return result

# Test RAG classification
test_tickets = [
    "Cannot login to my account",
    "Payment failed at checkout",
    "Package tracking shows stuck in transit",
    "The app keeps crashing when I open it",
]

print("RAG Classification Results:\n")
for ticket in test_tickets:
    result = classify_with_rag(ticket)
    print(f"Ticket: {ticket}")
    print(f"Classification: {result['classification']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print()

## 4. RAG Classification Pipeline

In [ ]:
# Simple TF-IDF based retriever
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create embeddings
documents = [doc[0] for doc in knowledge_base]
categories = [doc[1] for doc in knowledge_base]

vectorizer = TfidfVectorizer(stop_words='english')
doc_embeddings = vectorizer.fit_transform(documents)

def retrieve_documents(query: str, top_k: int = 2) -> List[Tuple[str, float]]:
    """Retrieve top-k documents for a query."""
    query_embedding = vectorizer.transform([query])
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append((documents[idx], float(similarities[idx])))
    return results

# Test retrieval
test_query = "I cannot login to my account"
results = retrieve_documents(test_query)
print(f"Query: '{test_query}'")
print(f"Retrieved documents:")
for doc, score in results:
    print(f"  Score: {score:.4f}")
    print(f"  Content: {doc[:80]}...")
    print()

## 3. Simple Retrieval System (Placeholder)

In [ ]:
# Create a simple knowledge base
knowledge_base = [
    ("Login Issues: User cannot access account. Solution: Reset password or check account status.",
     "Account"),
    ("Payment Errors: Transaction failed. Solution: Verify payment method and try again.",
     "Billing"),
    ("Shipping Delays: Package not arrived. Solution: Check tracking status with carrier.",
     "Shipping"),
    ("Product Quality: Item defective. Solution: Request replacement or refund.",
     "Product"),
    ("Technical Errors: Application crashed. Solution: Update app or clear cache.",
     "Technical"),
]

print(f"Knowledge base size: {len(knowledge_base)} documents")
for i, (doc, cat) in enumerate(knowledge_base):
    print(f"  {i}: {cat}")

## 2. Prepare Knowledge Base

## 1. RAG System Architecture

The RAG system consists of three main components:

1. **Retriever**: Finds relevant documents from a knowledge base using embedding similarity
2. **Prompt Engineering**: Creates contextual prompts using retrieved documents
3. **Generator**: Uses LLM to generate predictions based on context

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
import sys
sys.path.append('..')

# Try to import transformers and faiss for RAG
try:
    from transformers import AutoTokenizer, AutoModel
    print("Transformers library available")
except ImportError:
    print("Transformers not installed - install with: pip install transformers")

try:
    import faiss
    print("FAISS library available")
except ImportError:
    print("FAISS not installed - install with: pip install faiss-cpu")

# RAG System Experiments

This notebook demonstrates RAG (Retrieval-Augmented Generation) system experiments combining retrieval with transformers for ticket classification.